# Comprehensions and Generators

Comprehensions are Python's concise syntax for creating collections. Generators let you work with large sequences without loading everything into memory. Together, they're essential tools for writing clean, efficient data processing code.

## Learning Objectives

- Write list, dict, and set comprehensions fluently
- Know when NOT to use comprehensions (readability over cleverness)
- Understand the memory difference between comprehensions and generators
- Create generator functions with `yield`
- Measure memory usage to make informed decisions

In [ ]:
import sys
import matplotlib.pyplot as plt

## List Comprehensions

List comprehensions provide a concise way to create lists. The basic syntax is:

```python
[expression for item in iterable if condition]
```

This is equivalent to:
```python
result = []
for item in iterable:
    if condition:
        result.append(expression)
```

In [ ]:
# Basic list comprehension
squares = [x**2 for x in range(10)]
print(f"Squares: {squares}")

# With condition (filter)
even_squares = [x**2 for x in range(10) if x % 2 == 0]
print(f"Even squares: {even_squares}")

# Transform strings
words = ['Hello', 'World', 'Python']
upper_words = [w.upper() for w in words]
print(f"Upper: {upper_words}")

In [ ]:
# Multiple conditions
numbers = range(1, 31)
divisible_by_3_and_5 = [n for n in numbers if n % 3 == 0 and n % 5 == 0]
print(f"Divisible by 3 and 5: {divisible_by_3_and_5}")

# Conditional expression (ternary) in the output
labels = ['even' if x % 2 == 0 else 'odd' for x in range(6)]
print(f"Labels: {labels}")

## Dict and Set Comprehensions

Same concept, different brackets:
- List: `[expr for ...]`  
- Dict: `{key: value for ...}`  
- Set: `{expr for ...}`

In [ ]:
# Dict comprehension
names = ['alice', 'bob', 'charlie']
name_lengths = {name: len(name) for name in names}
print(f"Name lengths: {name_lengths}")

# Dict comprehension with condition
long_names = {name: len(name) for name in names if len(name) > 3}
print(f"Long names: {long_names}")

# Inverting a dictionary
original = {'a': 1, 'b': 2, 'c': 3}
inverted = {v: k for k, v in original.items()}
print(f"Inverted: {inverted}")

In [ ]:
# Set comprehension - automatically removes duplicates
text = "hello world"
unique_chars = {char for char in text if char != ' '}
print(f"Unique chars: {unique_chars}")

# Get unique first letters
words = ['apple', 'banana', 'apricot', 'blueberry', 'avocado']
first_letters = {w[0] for w in words}
print(f"First letters: {first_letters}")

## Nested Comprehensions

Comprehensions can be nested, but use them sparingly—readability matters more than brevity.

In [ ]:
# Flatten a 2D list
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

# Nested comprehension (read: for row in matrix, for num in row)
flattened = [num for row in matrix for num in row]
print(f"Flattened: {flattened}")

# Create a multiplication table
mult_table = [[i * j for j in range(1, 6)] for i in range(1, 6)]
print("Multiplication table:")
for row in mult_table:
    print(row)

In [ ]:
# Cartesian product
colors = ['red', 'green', 'blue']
sizes = ['S', 'M', 'L']

combinations = [(color, size) for color in colors for size in sizes]
print(f"Combinations: {combinations}")

> **Gotcha: When NOT to use comprehensions**  
> If a comprehension takes more than one line or requires mental parsing, use a regular for-loop. Readability counts!

In [ ]:
# BAD: Too complex - hard to understand at a glance
# result = [[cell * 2 for cell in row if cell > 0] for row in matrix if sum(row) > 10]

# GOOD: Use explicit loops for complex logic
result = []
for row in matrix:
    if sum(row) > 10:
        doubled_positives = []
        for cell in row:
            if cell > 0:
                doubled_positives.append(cell * 2)
        result.append(doubled_positives)
        
print(f"Result: {result}")

## Generator Expressions

Generator expressions look like list comprehensions but use parentheses. They produce values lazily—one at a time, on demand—rather than creating the entire list in memory.

In [ ]:
# List comprehension - creates list immediately
list_comp = [x**2 for x in range(5)]
print(f"List: {list_comp}, type: {type(list_comp)}")

# Generator expression - creates generator object
gen_exp = (x**2 for x in range(5))
print(f"Generator: {gen_exp}, type: {type(gen_exp)}")

# Iterate to get values
print(f"Generator values: {list(gen_exp)}")

In [ ]:
# Memory comparison
n = 1_000_000

# List stores all values in memory
list_size = sys.getsizeof([x for x in range(n)])

# Generator only stores the iteration state
gen_size = sys.getsizeof(x for x in range(n))

print(f"List size: {list_size:,} bytes ({list_size / 1024 / 1024:.2f} MB)")
print(f"Generator size: {gen_size} bytes")
print(f"Ratio: {list_size / gen_size:.0f}x")

> **Gotcha: Generators are exhausted after one use**  
> Once you iterate through a generator, it's empty. You can't reuse it.

In [ ]:
gen = (x for x in range(3))

print("First iteration:", list(gen))  # [0, 1, 2]
print("Second iteration:", list(gen))  # [] - exhausted!

## Generator Functions with `yield`

For more complex logic, use generator functions. The `yield` keyword pauses the function and returns a value; the function resumes from there on the next call.

In [ ]:
# Simple generator function
def count_up_to(n):
    """Generate numbers from 1 to n."""
    i = 1
    while i <= n:
        yield i
        i += 1

# Use the generator
for num in count_up_to(5):
    print(num, end=' ')
print()

In [ ]:
# Fibonacci generator - infinite sequence!
def fibonacci():
    """Generate Fibonacci numbers indefinitely."""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

# Get first 10 Fibonacci numbers
fib_gen = fibonacci()
first_10_fib = [next(fib_gen) for _ in range(10)]
print(f"First 10 Fibonacci: {first_10_fib}")

In [ ]:
# Practical example: reading large files line by line
def read_large_file(file_path):
    """Read a file line by line without loading all into memory."""
    with open(file_path, 'r') as f:
        for line in f:
            yield line.strip()

# This would be used like:
# for line in read_large_file('huge_data.csv'):
#     process(line)  # Only one line in memory at a time

In [ ]:
# Generator for batching data
def batch(iterable, size):
    """Yield batches of the given size from an iterable."""
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == size:
            yield batch
            batch = []
    if batch:  # Don't forget the last partial batch
        yield batch

# Example: process data in batches of 3
data = range(10)
for b in batch(data, 3):
    print(f"Processing batch: {b}")

## Memory Comparison Visualization

In [ ]:
# Measure memory for different sizes
sizes = [100, 1000, 10000, 100000, 1000000]
list_sizes = []
gen_sizes = []

for n in sizes:
    # List comprehension
    list_mem = sys.getsizeof([x for x in range(n)])
    list_sizes.append(list_mem / 1024)  # Convert to KB
    
    # Generator expression
    gen_mem = sys.getsizeof(x for x in range(n))
    gen_sizes.append(gen_mem / 1024)  # Convert to KB

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(sizes))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], list_sizes, width, label='List Comprehension', color='#e74c3c')
bars2 = ax.bar([i + width/2 for i in x], gen_sizes, width, label='Generator Expression', color='#2ecc71')

ax.set_xlabel('Number of Elements')
ax.set_ylabel('Memory Usage (KB)')
ax.set_title('Memory Usage: List vs Generator\n(Lower is Better)')
ax.set_xticks(x)
ax.set_xticklabels([f'{s:,}' for s in sizes])
ax.legend()
ax.set_yscale('log')

plt.tight_layout()
plt.show()

print(f"\nAt 1M elements:")
print(f"  List: {list_sizes[-1]:.2f} KB")
print(f"  Generator: {gen_sizes[-1]:.4f} KB")
print(f"  Ratio: {list_sizes[-1] / gen_sizes[-1]:.0f}x more memory for list!")

## Refactoring Challenge: Bad Code → Comprehension

Here's code that works but could be cleaner. Rewrite it using comprehensions.

In [ ]:
# Original: Extract emails from user data
users = [
    {'name': 'Alice', 'email': 'alice@example.com', 'active': True},
    {'name': 'Bob', 'email': 'bob@example.com', 'active': False},
    {'name': 'Charlie', 'email': 'charlie@example.com', 'active': True},
    {'name': 'Diana', 'email': 'diana@example.com', 'active': True},
]

# BAD: Verbose loop
active_emails_bad = []
for user in users:
    if user['active']:
        active_emails_bad.append(user['email'])
print(f"Bad version: {active_emails_bad}")

# GOOD: List comprehension
active_emails_good = [user['email'] for user in users if user['active']]
print(f"Good version: {active_emails_good}")

In [ ]:
# Original: Create a mapping of names to emails for active users

# BAD: Verbose loop
name_to_email_bad = {}
for user in users:
    if user['active']:
        name_to_email_bad[user['name']] = user['email']
print(f"Bad version: {name_to_email_bad}")

# GOOD: Dict comprehension
name_to_email_good = {user['name']: user['email'] for user in users if user['active']}
print(f"Good version: {name_to_email_good}")

## Exercises

### Exercise 1: List Comprehension Practice

Given a list of integers, create a new list containing only the squares of odd numbers greater than 5.

In [ ]:
numbers = [1, 4, 7, 2, 9, 3, 12, 8, 11, 5, 6]

# Expected output: [49, 81, 121] (squares of 7, 9, 11)
# YOUR CODE HERE

### Exercise 2: Dict Comprehension

Create a dictionary mapping each word in the sentence to its length, but only for words longer than 3 characters.

In [ ]:
sentence = "The quick brown fox jumps over the lazy dog"

# Expected: {'quick': 5, 'brown': 5, 'jumps': 5, 'over': 4, 'lazy': 4}
# YOUR CODE HERE

### Exercise 3: Generator Function

Write a generator function that yields prime numbers up to a given limit.

In [ ]:
def primes_up_to(n):
    """Generate prime numbers up to n."""
    # YOUR CODE HERE
    pass

# Test: should print 2, 3, 5, 7, 11, 13, 17, 19, 23, 29
# print(list(primes_up_to(30)))

### Exercise 4: Flatten and Transform

Given a nested list of transaction amounts by day, use a comprehension to flatten it and calculate the total, but only include positive amounts.

In [ ]:
transactions = [
    [100, -50, 200],   # Day 1
    [150, -20, -30],   # Day 2
    [80, 120, -10],    # Day 3
]

# 1. Flatten to a single list of all positive transactions
# Expected: [100, 200, 150, 80, 120]
# YOUR CODE HERE

# 2. Calculate the sum of all positive transactions using a generator expression
# YOUR CODE HERE